# Pipelines de Pré-processamento - Parte 2
## Data Leakage e ColumnTransformer

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 10 - Parte 2 de 2
---


## 1. Data Leakage e Como Prevenir

### 1.1 O que é Data Leakage?

Data leakage (vazamento de dados) ocorre quando informações que não estariam disponíveis no momento da predição são usadas para treinar um modelo. Esse vazamento pode fazer com que o modelo apresente um desempenho artificialmente alto durante o treinamento e validação, mas falhe significativamente quando aplicado a novos dados.

O data leakage é um dos erros mais comuns e perigosos em projetos de ciência de dados, pois pode:

- Criar uma falsa sensação de bom desempenho do modelo
- Levar a decisões de negócios incorretas
- Desperdiçar recursos em modelos que não funcionarão em produção
- Ser difícil de detectar, especialmente em pipelines complexos

### 1.2 Tipos de Data Leakage

### 1. Leakage na preparação dos dados

Ocorre quando as transformações de dados são aplicadas antes da divisão entre treino e teste:

data-leakage-prep__.svg

### 2. Leakage de target

Ocorre quando variáveis altamente correlacionadas com o target (que não estariam disponíveis na produção) são incluídas no treinamento:

target-leakage__.svg

### 3. Leakage de grupo

Ocorre quando dados relacionados a um mesmo grupo (ex: mesmo paciente, cliente ou entidade) estão presentes tanto no treino quanto no teste:

group-leakage__.svg

### 1.3 Como os Pipelines Evitam Data Leakage

Os pipelines do scikit-learn ajudam a prevenir o data leakage de várias formas:

1. **Separação clara entre fit e transform**: O método `fit` é aplicado apenas aos dados de treino, enquanto `transform` pode ser aplicado a qualquer conjunto.

2. **Encapsulamento das transformações**: Todas as etapas de pré-processamento ficam dentro de um único objeto, reduzindo a chance de erros.

3. **Aplicação automática da ordem correta**: As transformações são sempre aplicadas na mesma sequência.

4. **Fácil integração com validação cruzada**: Os pipelines podem ser usados diretamente em `cross_val_score` e similares.

Vamos ver um exemplo de como o data leakage pode ocorrer e como os pipelines ajudam a evitá-lo:

In [ ]:
# @title
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# MODIFICAÇÃO 1: Adicionando outliers artificiais ao dataset
# Isso fará com que a normalização seja muito afetada pelo vazamento
np.random.seed(42)

# Identificando um subconjunto de dados para ser "teste"
test_mask = np.zeros(len(X), dtype=bool)
test_mask[np.random.choice(len(X), size=int(len(X)*0.3), replace=False)] = True

# Adicionando outliers extremos apenas nas amostras que serão de teste
outlier_scale = 10  # Fator de escala para os outliers
for col in X.columns[:5]:  # Modificando apenas algumas colunas
    # Multiplicando os valores no conjunto de teste por um fator grande
    X.loc[test_mask, col] = X.loc[test_mask, col] * outlier_scale

# MODIFICAÇÃO 2: Adicionando valores ausentes de forma não aleatória
# Isso tornará a imputação mais sensível ao vazamento
missing_proportion = 0.2
for col in X.columns[5:10]:  # Escolhendo outras colunas para valores ausentes
    # Criando valores ausentes nos menores valores da coluna
    threshold = X[col].quantile(missing_proportion)
    X.loc[(X[col] <= threshold) & ~test_mask, col] = np.nan

print(f"Dataset modificado: {X.shape[0]} amostras, {X.shape[1]} features")
print(f"Valores ausentes: {X.isna().sum().sum()}")

print(f"Dataset carregado: {X.shape[0]} amostras, {X.shape[1]} features")
print(f"Valores ausentes: {X.isna().sum().sum()}")

### 1.4 Demonstração: Abordagem Incorreta

Na abordagem a seguir, realizamos a normalização antes da divisão treino/teste, o que causa data leakage:

In [ ]:
# ABORDAGEM COM DATA LEAKAGE: Normalização antes da divisão treino/teste

# 1. Preenchendo valores ausentes com a média
X_filled = X.fillna(X.mean())

# 2. Normalizando todos os dados antes da divisão (ERRO - DATA LEAKAGE!)
scaler = StandardScaler()
X_scaled_full = scaler.fit_transform(X_filled)

# 3. Agora dividindo em treino e teste
X_train_leak, X_test_leak, y_train_leak, y_test_leak = \
    train_test_split(X_scaled_full, y, test_size=0.3, random_state=42)

# 4. Treinando o modelo
model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_train_leak, y_train_leak)

# 5. Avaliando no conjunto de teste
y_pred_leak = model_leak.predict(X_test_leak)
accuracy_leak = accuracy_score(y_test_leak, y_pred_leak)

print(f"Acurácia com data leakage: {accuracy_leak:.4f}")

O problema da abordagem acima é que o StandardScaler utilizou informações de todos os dados, incluindo os outliers do conjunto de teste, para normalizar o conjunto. Em um cenário real, não teríamos acesso a essas informações no momento da predição.

### 1.5 Abordagem Correta

In [ ]:
# ABORDAGEM CORRETA: Divisão primeiro, normalização depois

# 1. Primeiro dividimos em treino e teste
X_train_correct, X_test_correct, y_train_correct, y_test_correct = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 2. Processamento manual separado para treino
# Preenchendo valores ausentes apenas com informações do treino
imputer_mean = X_train_correct.mean()
X_train_filled = X_train_correct.fillna(imputer_mean)

# Normalizando apenas com informações do treino
scaler_correct = StandardScaler()
X_train_scaled = scaler_correct.fit_transform(X_train_filled)

# 3. Aplicando as mesmas transformações ao teste (usando parâmetros do treino)
X_test_filled = X_test_correct.fillna(imputer_mean)
X_test_scaled = scaler_correct.transform(X_test_filled)

# 4. Treinando o modelo
model_correct = LogisticRegression(max_iter=1000)
model_correct.fit(X_train_scaled, y_train_correct)

# 5. Avaliando no conjunto de teste
y_pred_correct = model_correct.predict(X_test_scaled)
accuracy_correct = accuracy_score(y_test_correct, y_pred_correct)

print(f"Acurácia sem data leakage (manual): {accuracy_correct:.4f}")

### 1.6 Usando Pipelines para Prevenir

A abordagem correta mostrada acima é trabalhosa e propensa a erros. Os pipelines automatizam esse processo de forma segura:

In [ ]:
# ABORDAGEM COM PIPELINE: Mais simples e segura
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Primeiro dividimos em treino e teste (sempre o primeiro passo!)
X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe = \
    train_test_split(X, y, test_size=0.3, random_state=42
)

# 2. Definindo o pipeline de pré-processamento e modelo
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # Usa SimpleImputer do scikit-learn
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# 3. Treinando o pipeline (não é mais necessário passar parâmetros específicos)
pipeline.fit(X_train_pipe, y_train_pipe)

# 4. Avaliando no conjunto de teste
y_pred_pipe = pipeline.predict(X_test_pipe)
accuracy_pipe = accuracy_score(y_test_pipe, y_pred_pipe)

print(f"Acurácia com pipeline: {accuracy_pipe:.4f}")

### 1.7 Comparando as Abordagens

In [ ]:
# @title
# Comparando as acurácias das três abordagens
accuracies = [accuracy_leak, accuracy_correct, accuracy_pipe]
labels = ['Com Data Leakage', 'Sem Leakage (Manual)', 'Com Pipeline']

plt.figure(figsize=(10, 6))
bars = plt.bar(labels, accuracies, color=['#ff9999', '#66b3ff', '#99ff99'])
plt.ylim(0.0, 1.0)  # Ajustando para melhor visualização
plt.title('Comparação de Acurácia entre Abordagens')
plt.ylabel('Acurácia')

# Adicionando os valores de acurácia em cada barra
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height - 0.005,
            f'{height:.4f}', ha='center', va='top', color='white', fontweight='bold')

# Alternativa ao tight_layout que evita o aviso
plt.subplots_adjust(bottom=0.15, top=0.9)
plt.show()

### 1.8 Target Leakage

Target leakage ocorre quando uma variável altamente correlacionada com o target, mas que não estaria disponível no momento da predição, é incluída no modelo.

Vamos criar um exemplo artificial adicionando uma variável com "vazamento":

`X_with_leak['target_leak'] = y + np.random.normal(0, 0.9, size=len(y))`


In [ ]:
# @title
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Criando um exemplo de target leakage
np.random.seed(42)

# Adicionando uma feature com alta correlação com o target (vazamento)
X_with_leak = X.copy()
X_with_leak['target_leak'] = y + np.random.normal(0, 0.9, size=len(y))

# Dividindo em treino e teste
X_train_tl, X_test_tl, y_train_tl, y_test_tl = train_test_split(
    X_with_leak, y, test_size=0.3, random_state=42
)

# Criando um pipeline básico COM VAZAMENTO
pipeline_with_leak = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000,C=1e6))
])

# Treinando o pipeline
pipeline_with_leak.fit(X_train_tl, y_train_tl)

# Avaliando no conjunto de teste
y_pred_tl = pipeline_with_leak.predict(X_test_tl)
accuracy_tl = accuracy_score(y_test_tl, y_pred_tl)

# Comparando com o pipeline sem vazamento
pipeline_no_leak = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000,C=1e6))
])

# Removendo a coluna de vazamento
X_train_no_tl = X_train_tl.drop('target_leak', axis=1)
X_test_no_tl = X_test_tl.drop('target_leak', axis=1)

# Treinando o pipeline sem vazamento
pipeline_no_leak.fit(X_train_no_tl, y_train_tl)

# Avaliando no conjunto de teste
y_pred_no_tl = pipeline_no_leak.predict(X_test_no_tl)
accuracy_no_tl = accuracy_score(y_test_tl, y_pred_no_tl)

print(f"Acurácia com target leakage: {accuracy_tl:.4f}")
print(f"Acurácia sem target leakage: {accuracy_no_tl:.4f}")

### 1.9 Boas Práticas

1. **Sempre divida os dados primeiro**: Separar treino/teste/validação deve ser a primeira etapa.

2. **Use pipelines**: Automatizam a aplicação correta das transformações.

3. **Verifique correlações suspeitas**: Features com correlação muito alta com o target podem indicar vazamento.

4. **Pense cronologicamente**: Considere quais informações estariam disponíveis no momento da predição.

5. **Validação temporal**: Para dados temporais, valide usando períodos futuros.

6. **Validação estratificada por grupo**: Para evitar leakage de grupo, faça divisões que mantenham entidades relacionadas juntas.

7. **Documentação clara**: Documente quais transformações foram aplicadas e em que ordem.

8. **Revisão por pares**: Peça a outros cientistas de dados para revisar seu pipeline.



## 2. ColumnTransformer

### 2.1 O Desafio de Múltiplos Tipos de Dados

Na prática, a maioria dos conjuntos de dados contém uma mistura de diferentes tipos de variáveis:

- **Numéricas**: inteiros e floats (ex: idade, salário, temperatura)
- **Categóricas**: nominais e ordinais (ex: gênero, cidade, nível de escolaridade)
- **Textuais**: strings livres (ex: comentários, descrições de produtos)
- **Temporais**: datas e horas (ex: data de nascimento, timestamp)

Cada tipo de dado requer transformações específicas:

| Tipo de Dado | Transformações Típicas |
|--------------|------------------------|
| Numérico     | Imputação de valores ausentes, normalização, escalonamento |
| Categórico   | Encoding (one-hot, label, target, etc.), imputação |
| Textual      | Tokenização, remoção de stopwords, TF-IDF, embeddings |
| Temporal     | Extração de características (dia da semana, mês, etc.), duração, periodicidade |

Como podemos aplicar diferentes transformações a diferentes colunas em um pipeline?

### 2.2 Introdução ao ColumnTransformer

O `ColumnTransformer` do scikit-learn permite aplicar diferentes transformações a diferentes subconjuntos de características em um único passo. Ele funciona especificando:

1. As transformações a serem aplicadas
2. As colunas às quais cada transformação se aplica
3. Opcionalmente, como tratar colunas não especificadas

A sintaxe básica é:

```python
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('nome1', transformador1, colunas1),
        ('nome2', transformador2, colunas2),
        ...
    ],
    remainder='drop'  # ou 'passthrough' para manter colunas não especificadas
)
```

Isso permite criar pipelines complexos que tratam diferentes tipos de dados de forma apropriada.

In [ ]:
# @title Criando um dataset sintético
# Importando bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# Criando um dataset de exemplo com diferentes tipos de dados
np.random.seed(42)

# Tamanho do dataset
n = 1000

# Criando features numéricas - não use astype(int) antes de adicionar NaN
idade = np.random.normal(35, 10, n).round().astype(float)  # Mantém como float para poder ter NaN
salario = np.random.lognormal(10, 1, n)
score_credito = np.random.normal(650, 100, n).round().astype(float)  # Também como float


# Criando features categóricas
genero = np.random.choice(['M', 'F'], n)
estado_civil = np.random.choice(['Solteiro', 'Casado', 'Divorciado', 'Viúvo'], n)
nivel_educacao = np.random.choice(['Ensino Fundamental', 'Ensino Médio', 'Graduação', 'Pós-graduação'], n)

# Criando feature textual simples
textos = []
for i in range(n):
    opcoes = [
        "Cliente interessado em empréstimo pessoal.",
        "Cliente solicitou informações sobre investimentos.",
        "Cliente com histórico de inadimplência.",
        "Cliente fiel há muitos anos.",
        "Cliente novo sem histórico."
    ]
    textos.append(np.random.choice(opcoes))

# Criando o target (aprovação de crédito)
prob_aprovacao = np.zeros(n)

for i in range(n):
    prob_aprovacao[i] = 1 / (1 + np.exp(-(
        (score_credito[i] - 600) / 100 +  # Score de crédito positivo
        (salario[i] - 50000) / 50000 -    # Salário positivo
        (1 if estado_civil[i] == 'Divorciado' else 0) * 0.5 -  # Ser divorciado é levemente negativo
        (1 if "inadimplência" in textos[i] else 0) * 2      # Histórico de inadimplência é muito negativo
    )))

aprovado = (np.random.random(n) < prob_aprovacao).astype(int)

# Adicionando alguns valores ausentes aleatoriamente
idade[np.random.choice(n, int(n*0.05), replace=False)] = np.nan
salario[np.random.choice(n, int(n*0.1), replace=False)] = np.nan
genero[np.random.choice(n, int(n*0.03), replace=False)] = None
nivel_educacao[np.random.choice(n, int(n*0.08), replace=False)] = None

# Criando o DataFrame
df = pd.DataFrame({
    'idade': idade,
    'salario': salario,
    'score_credito': score_credito,
    'genero': genero,
    'estado_civil': estado_civil,
    'nivel_educacao': nivel_educacao,
    'comentario': textos,
    'aprovado': aprovado
})

# Visualizando o dataset
print("Primeiras linhas do dataset:")
print(df.head())

print("\nInformações do dataset:")
print(df.info())

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

### 2.3 Definindo Transformações

Agora vamos definir transformações específicas para cada tipo de variável:

1. **Features numéricas**: imputação de valores ausentes + padronização
2. **Features categóricas**: imputação + one-hot encoding
3. **Features textuais**: TF-IDF vectorizer

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# Definindo as colunas por tipo
numeric_features = ['idade', 'salario', 'score_credito']
categorical_features = ['genero', 'estado_civil', 'nivel_educacao']
text_features = ['comentario']

# Pipeline para features numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para features categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


# Transformador de texto para lidar adequadamente com o formato dos dados
class TextTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=None):
        self.max_features = max_features
        self.vectorizer = TfidfVectorizer(max_features=max_features)

    def fit(self, X, y=None):
        # Garantir que X é uma série ou lista de strings
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]  # Pegar a primeira coluna se for DataFrame

        self.vectorizer.fit(X)
        return self

    def transform(self, X):
        # Garantir que X é uma série ou lista de strings
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]  # Pegar a primeira coluna se for DataFrame

        return self.vectorizer.transform(X)

    def get_feature_names_out(self):
        try:
            return self.vectorizer.get_feature_names_out()
        except:
            # Fallback para versões mais antigas do scikit-learn
            return self.vectorizer.get_feature_names()

# Combinando todos os transformadores com ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', TextTransformer(), text_features)  # Usando o novo transformador diretamente
    ]
)

# Visualizando a estrutura do preprocessor
print("Estrutura do preprocessor:")
print(preprocessor)

### 2.4 Pipeline Completo

Vamos integrar o preprocessor em um pipeline completo que inclui também a etapa de modelagem:

In [ ]:
# Criando o pipeline completo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])


# Separando o target do dataset
X = df.drop('aprovado', axis=1)
y = df['aprovado']

# Dividindo os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Ajustando o pipeline nos dados de treino
pipeline.fit(X_train, y_train)

# Fazendo previsões
y_pred = pipeline.predict(X_test)

# Avaliando o desempenho
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### 2.5 Engenharia de Features

Além das transformações básicas, podemos incluir etapas de engenharia de features no pipeline. Vamos adicionar algumas transformações mais complexas:

1. Para variáveis numéricas: transformações polinomiais
2. Para variáveis categóricas: encoding ordinal para variáveis com ordem
3. Para texto: análise de sentimento como feature adicional

In [ ]:
# @title


# Transformação polinomial para features numéricas
numeric_transformer_poly = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=5))  # Selecionamos as 5 melhores features
])

# Transformação ordinal para nível de educação (que tem uma ordem intrínseca)
# Definindo a ordem das categorias
education_categories = ['Ensino Fundamental', 'Ensino Médio', 'Graduação', 'Pós-graduação']

# Função customizada para pré-processamento que garante que None seja tratado corretamente
def clean_education(X):
    # Converte explicitamente None para np.nan (que o SimpleImputer pode tratar)
    if isinstance(X, pd.DataFrame) or isinstance(X, pd.Series):
        return X.replace({None: np.nan})
    else:
        # Para arrays NumPy
        X_copy = X.copy()
        mask = np.array([x is None for x in X_copy.flatten()]).reshape(X_copy.shape)
        X_copy[mask] = np.nan
        return X_copy

# Pipeline para variáveis categóricas com encoding apropriado
categorical_transformer_mixed = ColumnTransformer(
    transformers=[
        ('onehot', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), ['genero', 'estado_civil']),

        ('ordinal', Pipeline(steps=[
            ('cleaner', FunctionTransformer(clean_education)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(categories=[education_categories], handle_unknown='use_encoded_value', unknown_value=-1))
        ]), ['nivel_educacao'])
    ]
)

# Classe TextTransformerEnhanced para lidar adequadamente com o formato dos dados
class TextTransformerEnhanced(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=10):
        self.max_features = max_features
        self.vectorizer = TfidfVectorizer(max_features=max_features)

    def fit(self, X, y=None):
        # Tratar DataFrame ou Series
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]

        # Lidar com valores None - converter para string vazia
        X_clean = ['' if x is None else str(x) for x in X]

        self.vectorizer.fit(X_clean)
        return self

    def transform(self, X):
        # Tratar DataFrame ou Series
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]

        # Lidar com valores None - converter para string vazia
        X_clean = ['' if x is None else str(x) for x in X]

        # Calcular sentimentos
        sentiment = self._get_sentiment(X_clean)

        # Transformar texto usando TF-IDF
        tfidf_features = self.vectorizer.transform(X_clean)

        # Concatenar horizontalmente (sparse + dense)
        from scipy.sparse import hstack, csr_matrix

        # Garantir que ambos sejam matrizes esparsas com o mesmo número de linhas
        if not isinstance(sentiment, csr_matrix):
            sentiment = csr_matrix(sentiment)

        return hstack([tfidf_features, sentiment])

    def _get_sentiment(self, texts):
        scores = []
        for text in texts:
            # IMPORTANTE: text já deve ser string aqui, mas vamos garantir
            if text is None:
                text = ''

            text = str(text)  # Garantir que é string

            # Implementação muito simplificada
            positive_words = ['interessado', 'informações', 'fiel', 'muitos anos']
            negative_words = ['inadimplência', 'sem histórico']

            score = 0
            for word in positive_words:
                if word in text.lower():  # Aqui text.lower() estava causando erro se text fosse None
                    score += 1
            for word in negative_words:
                if word in text.lower():
                    score -= 1

            scores.append([score])

        # Retornar como matriz NumPy
        return np.array(scores)

# Atualizando a função de teste de amostra
def test_sample(pipeline, sample_new):
    try:
        # Primeiro, tente só o preprocessor
        preprocessor = pipeline.named_steps['preprocessor']
        print("\nTestando preprocessor:")
        transformed = preprocessor.transform(sample_new)
        print("Preprocessor funcionou! Shape dos dados transformados:", transformed.shape)

        # Se o preprocessor funcionou, tente o pipeline completo
        print("\nTestando pipeline completo:")
        preds = pipeline.predict(sample_new)
        probs = pipeline.predict_proba(sample_new)

        print("Predições:", preds)
        print("Probabilidades de aprovação:", probs[:, 1])

    except Exception as e:
        print(f"\nErro: {e}")

        # Tentar identificar onde o erro está ocorrendo
        for name, step in pipeline.named_steps.items():
            print(f"Testando o passo: {name}")
            try:
                if name == 'preprocessor':
                    step.transform(sample_new)
                    print(f"Passo {name} executado com sucesso.")
                else:
                    # Para o classificador, precisamos de dados já transformados
                    X_transformed = pipeline.named_steps['preprocessor'].transform(sample_new)
                    step.predict(X_transformed)
                    print(f"Passo {name} executado com sucesso.")
            except Exception as e:
                print(f"Erro no passo {name}: {e}")

# Criando o preprocessador completo
preprocessor_enhanced = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_poly, numeric_features),
        ('cat', categorical_transformer_mixed, categorical_features),
        ('text', TextTransformerEnhanced(max_features=10), text_features)
    ]
)

# Criando o pipeline final
pipeline_enhanced = Pipeline(steps=[
    ('preprocessor', preprocessor_enhanced),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_enhanced.fit(X_train, y_train)

### 2.6 Valores Ausentes na Predição

Um desafio comum é lidar com valores ausentes ou categorias desconhecidas no momento da predição. Vamos ver como o pipeline lida com isso:

In [ ]:
# @title
# Criando uma amostra com valores ausentes e categorias novas
sample_new = pd.DataFrame({
    'idade': [25, np.nan, 40],
    'salario': [np.nan, 80000, 120000],
    'score_credito': [720, 680, np.nan],
    'genero': ['M', 'X', None],  # 'X' é uma categoria nova
    'estado_civil': ['Solteiro', None, 'Separado'],  # 'Separado' é uma categoria nova
    'nivel_educacao': [None, 'Doutorado', 'Ensino Médio'],  # 'Doutorado' é uma categoria nova
    'comentario': [
        "Cliente com excelente histórico de crédito.",  # texto novo
        None,  # valor ausente
        "Cliente solicitou informações sobre investimentos."
    ]
})

print("Amostra com valores ausentes e categorias novas:")
print(sample_new)


# Agora teste o pipeline já treinado
try:
    # Usar diretamente o pipeline completo para predições
    preds = pipeline_enhanced.predict(sample_new)
    probs = pipeline_enhanced.predict_proba(sample_new)

    print("\nPredições:", preds)
    print("Probabilidades de aprovação:", probs[:, 1])
    print("\nO pipeline lidou bem com valores ausentes e categorias novas!")

except Exception as e:
    print(f"\nErro ao fazer predições: {e}")

    # Se houve erro, vamos tentar entender melhor o que aconteceu
    for name, step in pipeline_enhanced.named_steps.items():
        print(f"\nVerificando o passo '{name}':")
        try:
            if hasattr(step, 'fit') and not hasattr(step, '_is_fitted'):
                print(f"  - O passo '{name}' não parece estar treinado ainda!")
            else:
                print(f"  - O passo '{name}' parece estar treinado.")
        except:
            print(f"  - Não foi possível verificar se o passo '{name}' está treinado.")


### 2.7 Salvando e Carregando Pipelines

Uma das grandes vantagens dos pipelines é a facilidade para salvar o modelo completo, incluindo todas as etapas de pré-processamento, e carregá-lo para uso posterior.

In [ ]:
import joblib
import os

# Criando um diretório para salvar o modelo (se ainda não existir)
if not os.path.exists('modelos'):
    os.makedirs('modelos')

# Salvando o pipeline completo
joblib.dump(pipeline, 'modelos/pipeline_credito.pkl')

# Carregando o pipeline
pipeline_loaded = joblib.load('modelos/pipeline_credito.pkl')

# Testando o pipeline carregado
y_pred_loaded = pipeline_loaded.predict(X_test)
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)

print(f"Acurácia do modelo carregado: {accuracy_loaded:.4f}")
print(f"Os resultados são idênticos? {np.array_equal(y_pred, y_pred_loaded)}")

## 🎯 Micro-atividade: Data Leakage Detective (10 min)

**Objetivo:** Identificar e corrigir data leakage em código

### Cenário:
Você recebeu o código abaixo de um colega. Ele afirma ter 95% de acurácia, mas você suspeita de data leakage.
> *Não precisa executar o código abaixo, só identifcar os probelmas!*

```python
# Código suspeito
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Carregar dados
X, y = load_data()  # Assuma que isso funciona

# Normalizar TODOS os dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Agora dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Treinar modelo
model = LogisticRegression()
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
print(f"Acurácia: {score:.2%}")
```

### Tarefas:
1. **Identifique** o problema de data leakage
2. **Explique** por que isso é problemático
3. **Corrija** o código usando um pipeline

### Dica:
Pense sobre que informações o StandardScaler "aprende" dos dados...

### 2.8 Dicas e Melhores Práticas

### 1. Organização e Nomenclatura
- Use nomes descritivos para suas transformações
- Agrupe transformações relacionadas em sub-pipelines
- Documente a finalidade de cada etapa do pipeline

### 2. Ajuste de Hiperparâmetros
- Use `GridSearchCV` ou `RandomizedSearchCV` com o pipeline completo
- Acesse parâmetros de etapas específicas usando a notação `nome_da_etapa__nome_do_parametro`
- Exemplo:
  ```python
  param_grid = {
      'preprocessor__num__imputer__strategy': ['mean', 'median'],
      'preprocessor__cat__onehot__handle_unknown': ['ignore', 'error'],
      'classifier__C': [0.1, 1.0, 10.0]
  }
  ```

### 3. Tratamento de Erros e Robustez
- Use `handle_unknown='ignore'` no `OneHotEncoder` para novas categorias
- Considere tratamento de valores ausentes explícito para cada tipo de coluna
- Valide seu pipeline com dados que simulam problemas comuns

### 4. Desempenho e Otimização
- Use `memory` no Pipeline para cache de resultados intermediários
- Para datasets grandes, considere transformadores incrementais
- Utilize `verbose=True` para monitorar o progresso

### 5. Extensão e Customização
- Crie transformadores personalizados herdando de `BaseEstimator` e `TransformerMixin`
- Implemente os métodos `fit`, `transform` e opcionalmente `fit_transform`
- Combine-os com transformadores do scikit-learn em pipelines híbridos



